# 04 · DistilBERT en Google Colab (Fase III)

*Fine-tuning* de `distilbert-base-uncased` sobre las **136 áreas** (sección 6.4). El modelo es
plano: `P(dominio)` se obtiene sumando las probabilidades de sus áreas, con la misma
decodificación jerárquica que los modelos clásicos (`clasificador_ml.seleccion`).

**Antes de ejecutar**

1. *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4).*
2. Subir a Google Drive, en `MyDrive/sicda/data/processed/`, los archivos
   `train.parquet`, `val.parquet`, `test.parquet` y `categorias.json` del repositorio
   (`data/processed/`).

**Resultado** (en `MyDrive/sicda/models/distilbert/<version>/`): `metadatos.json`,
`probabilidades_val.parquet`, `probabilidades_test.parquet` y `modelo/`. Copiar esa carpeta
a `models/distilbert/<version>/` del repositorio para el notebook 05 (`modelo/` es opcional,
~260 MB). El test **no** se evalúa aquí: solo se guardan sus probabilidades.

Duración aproximada en una T4: 30 a 40 minutos.

In [ ]:
!nvidia-smi -L
# Solo hace falta `normalizar`, `evaluacion` y `seleccion`; Colab ya trae pandas y scikit-learn.
!pip install -q --no-deps "git+https://github.com/ElkinAcevedo/sicda.git@develop#subdirectory=ml"

In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')

CARPETA = Path('/content/drive/MyDrive/sicda')
DATOS = CARPETA / 'data' / 'processed'
VERSION = datetime.now().strftime('%Y%m%d-%H%M')
SALIDA = CARPETA / 'models' / 'distilbert' / VERSION

# Igual que ml/configs/distilbert.yaml y la semilla de base.yaml.
CONFIG = {
    'modelo': 'distilbert-base-uncased',
    'max_longitud': 256,
    'epocas': 3,
    'tamano_lote': 32,
    'tasa_aprendizaje': 5e-5,
    'semilla': 42,
}
print('GPU:', torch.cuda.is_available(), '· salida:', SALIDA)

In [ ]:
from clasificador_ml.preprocesamiento import normalizar

particiones = {n: pd.read_parquet(DATOS / f'{n}.parquet') for n in ('train', 'val', 'test')}
categorias = json.loads((DATOS / 'categorias.json').read_text(encoding='utf-8'))
area_a_dominio = {a['id']: d['id'] for d in categorias['dominios'] for a in d['areas']}
areas = sorted(area_a_dominio)
assert areas == list(range(len(areas))), 'Se espera id_area = 0..N-1 para usarlo como etiqueta'
{n: len(df) for n, df in particiones.items()}, len(areas)

## Tokenización

DistilBERT usa solo `normalizar` (minúsculas, sin tildes, URL, correos, números ni puntuación);
no se lematiza ni se quitan *stopwords*, porque el modelo aprovecha el contexto.

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(CONFIG['semilla'])
tokenizador = AutoTokenizer.from_pretrained(CONFIG['modelo'])


def a_dataset(df):
    datos = Dataset.from_dict({
        'text': [normalizar(t) for t in df['texto']],
        'label': df['id_area'].tolist(),
    })
    return datos.map(
        lambda lote: tokenizador(lote['text'], truncation=True, max_length=CONFIG['max_longitud']),
        batched=True,
        remove_columns=['text'],
    )


conjuntos = {n: a_dataset(df) for n, df in particiones.items()}

## Entrenamiento

Se evalúa en validación al final de cada época y se conserva la de mayor F1-macro de área.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

modelo = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['modelo'],
    num_labels=len(areas),
    id2label={a: str(a) for a in areas},
    label2id={str(a): a for a in areas},
)


def metricas(prediccion):
    logits, etiquetas = prediccion
    predichas = logits.argmax(axis=-1)
    return {
        'accuracy': accuracy_score(etiquetas, predichas),
        'f1_macro': f1_score(etiquetas, predichas, average='macro'),
    }


argumentos = TrainingArguments(
    output_dir='/content/salida',
    num_train_epochs=CONFIG['epocas'],
    per_device_train_batch_size=CONFIG['tamano_lote'],
    per_device_eval_batch_size=64,
    learning_rate=CONFIG['tasa_aprendizaje'],
    weight_decay=0.01,
    # Calentamiento en el primer 10 % de los pasos. Se calcula a mano porque las versiones
    # recientes de transformers ya no aceptan `warmup_ratio`.
    warmup_steps=int(0.1 * CONFIG['epocas'] * -(-len(conjuntos['train']) // CONFIG['tamano_lote'])),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    seed=CONFIG['semilla'],
    report_to='none',
)
entrenador = Trainer(
    model=modelo,
    args=argumentos,
    train_dataset=conjuntos['train'],
    eval_dataset=conjuntos['val'],
    data_collator=DataCollatorWithPadding(tokenizador),
    compute_metrics=metricas,
)
inicio = time.time()
entrenador.train()
segundos_entrenamiento = time.time() - inicio
print(f'Entrenamiento: {segundos_entrenamiento / 60:.1f} min')

## Probabilidades de validación y test

In [ ]:
SALIDA.mkdir(parents=True, exist_ok=True)


def probabilidades(nombre):
    logits = entrenador.predict(conjuntos[nombre]).predictions
    p = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
    tabla = pd.DataFrame(p, columns=[f'p_{a}' for a in areas])
    tabla.insert(0, 'id_documento', particiones[nombre]['id_documento'].to_numpy())
    return tabla


probabilidad = {n: probabilidades(n) for n in ('val', 'test')}
for nombre, tabla in probabilidad.items():
    tabla.to_parquet(SALIDA / f'probabilidades_{nombre}.parquet', index=False)

In [ ]:
from clasificador_ml.evaluacion import evaluar, guardar_json
from clasificador_ml.seleccion import predicciones_desde_probabilidades

# Solo validación: el test se evalúa una única vez en el notebook 05.
val = particiones['val']
prediccion_val = predicciones_desde_probabilidades(probabilidad['val'], area_a_dominio)
metricas_val = evaluar(val['id_dominio'], prediccion_val['dominio_pred'],
                       val['id_area'], prediccion_val['area_pred'])
pd.DataFrame(metricas_val).round(4)

## Tiempo por resumen

La API corre en CPU, así que el tiempo que cuenta para la selección (RNF-01) es el de CPU.
Incluye `normalizar` y la tokenización.

In [ ]:
def medir(dispositivo, cantidad):
    red = entrenador.model.to(dispositivo).eval()
    textos = particiones['val']['texto'].head(cantidad + 1).tolist()
    tiempos = []
    with torch.no_grad():
        for i, texto in enumerate(textos):
            inicio = time.perf_counter()
            entrada = tokenizador(normalizar(texto), truncation=True,
                                  max_length=CONFIG['max_longitud'], return_tensors='pt')
            red(**entrada.to(dispositivo))
            if i:  # el primero es de calentamiento
                tiempos.append((time.perf_counter() - inicio) * 1000)
    return {'documentos': len(tiempos), 'media_ms': float(np.mean(tiempos)),
            'p95_ms': float(np.percentile(tiempos, 95)), 'max_ms': float(np.max(tiempos))}


tiempo_gpu = medir('cuda', 100) if torch.cuda.is_available() else None
tiempo_cpu = medir('cpu', 30)
tiempo_cpu, tiempo_gpu

In [ ]:
historial = [r for r in entrenador.state.log_history if 'eval_f1_macro' in r]
metadatos = {
    'algoritmo': 'distilbert',
    'version': VERSION,
    'fecha': datetime.now().isoformat(timespec='seconds'),
    'configuracion': CONFIG,
    'datos': {n: len(df) for n, df in particiones.items()},
    'tipo': 'plano',
    'metricas_validacion': metricas_val,
    'porcentaje_revision_validacion': float(prediccion_val['requiere_revision'].mean()),
    'tiempo_entrenamiento_s': round(segundos_entrenamiento, 1),
    'tiempo_prediccion': tiempo_cpu,
    'tiempo_prediccion_gpu': tiempo_gpu,
    'historial_validacion': historial,
}
guardar_json(metadatos, SALIDA / 'metadatos.json')
entrenador.save_model(str(SALIDA / 'modelo'))
tokenizador.save_pretrained(str(SALIDA / 'modelo'))
print('Listo:', SALIDA)

Copiar `MyDrive/sicda/models/distilbert/<version>/` a `models/distilbert/<version>/` del
repositorio y abrir el notebook 05.